In [ ]:
# ------------------------------------------------------------------------------
# Cell 1: Install Dependencies (Diffusers + SDNQ + FastAPI)
# ------------------------------------------------------------------------------
!pip install -q fastapi uvicorn nest_asyncio requests huggingface_hub python-multipart
!pip install -q diffusers transformers accelerate peft sentencepiece protobuf
!pip install -q sdnq

In [ ]:
# ------------------------------------------------------------------------------
# Cell 2: Create Headless API Script (diffusers_api.py)
# UPGRADED: INT8 Model + Manual Embedding Bypass for Infinite Prompts
# ------------------------------------------------------------------------------
code_content = r'''
import os
import gc
import json
import time
import uuid
import threading
import inspect
from pathlib import Path

import torch
import diffusers
from sdnq import SDNQConfig
from sdnq.common import use_torch_compile as triton_is_available
from sdnq.loader import apply_sdnq_options_to_model

from fastapi import FastAPI, Form, BackgroundTasks
from fastapi.responses import StreamingResponse
import uvicorn

OUTPUTS_DIR = "/kaggle/working/outputs"
Path(OUTPUTS_DIR).mkdir(parents=True, exist_ok=True)

print("[API] Booting SDNQ INT8 Model... (Takes ~2 mins to download/load)")

# Load Model - Switched to INT8!
try:
    pipe = diffusers.ZImagePipeline.from_pretrained("Disty0/Z-Image-Turbo-SDNQ-int8", torch_dtype=torch.bfloat16)
except AttributeError:
    pipe = diffusers.DiffusionPipeline.from_pretrained("Disty0/Z-Image-Turbo-SDNQ-int8", torch_dtype=torch.bfloat16)

if triton_is_available and (torch.cuda.is_available() or torch.xpu.is_available()):
    pipe.transformer = apply_sdnq_options_to_model(pipe.transformer, use_quantized_matmul=True)
    pipe.text_encoder = apply_sdnq_options_to_model(pipe.text_encoder, use_quantized_matmul=True)

# The most important line for 16GB VRAM stability:
pipe.enable_model_cpu_offload()

print("[API] ✅ INT8 Model loaded successfully with CPU Offloading.")

app = FastAPI(title="Z-Image-Turbo Diffusers API")
_lock = threading.Lock() # Ensures only one generation runs at a time

def process_workflow_bg(prompt, negative, steps, cfg, w, h, seed, max_seq_len, task_id):
    status_file = Path(OUTPUTS_DIR) / f"task_{task_id}.json"
    status_file.write_text(json.dumps({"status": "processing"}))

    with _lock:
        try:
            # Setup Seed
            gen = torch.manual_seed(seed) if seed else torch.Generator(device="cpu").manual_seed(int(uuid.uuid4().int & (2**31 - 1)))
            print(f"[API] Generating task {task_id} (Checking prompt length limits)...")
            
            call_params = inspect.signature(pipe.__call__).parameters
            
            # Strategy 1: Native support
            if "max_sequence_length" in call_params:
                print(f"[API] Using native max_sequence_length={max_seq_len}.")
                image = pipe(prompt=prompt, height=h, width=w, num_inference_steps=steps, guidance_scale=cfg, generator=gen, max_sequence_length=max_seq_len).images[0]
            
            # Strategy 2: Manual Embedding Bypass (Forces 512 tokens!)
            elif "prompt_embeds" in call_params and hasattr(pipe, "tokenizer") and hasattr(pipe, "text_encoder"):
                print(f"[API] Pipeline has a hardcoded limit! Manually forcing {max_seq_len} tokens...")
                device = pipe._execution_device if hasattr(pipe, "_execution_device") else "cuda"
                
                # Encode Positive Prompt manually
                text_inputs = pipe.tokenizer(prompt, padding="max_length", max_length=max_seq_len, truncation=True, return_attention_mask=True, return_tensors="pt")
                with torch.no_grad():
                    prompt_embeds = pipe.text_encoder(text_inputs.input_ids.to(device), attention_mask=text_inputs.attention_mask.to(device))[0]
                
                kwargs = {
                    "prompt_embeds": prompt_embeds,
                    "height": h, "width": w, "num_inference_steps": steps,
                    "guidance_scale": cfg, "generator": gen
                }
                if "prompt_attention_mask" in call_params:
                    kwargs["prompt_attention_mask"] = text_inputs.attention_mask.to(device)
                    
                # Encode Negative Prompt manually (Only needed if CFG > 1.0)
                if cfg > 1.0:
                    neg_inputs = pipe.tokenizer(negative, padding="max_length", max_length=max_seq_len, truncation=True, return_attention_mask=True, return_tensors="pt")
                    with torch.no_grad():
                        neg_embeds = pipe.text_encoder(neg_inputs.input_ids.to(device), attention_mask=neg_inputs.attention_mask.to(device))[0]
                    kwargs["negative_prompt_embeds"] = neg_embeds
                    if "negative_prompt_attention_mask" in call_params:
                        kwargs["negative_prompt_attention_mask"] = neg_inputs.attention_mask.to(device)
                        
                # Pass raw math directly to pipeline
                image = pipe(**kwargs).images[0]
                
            # Strategy 3: Absolute fallback
            else:
                print("[API] ⚠️ Cannot bypass prompt limit. Using default pipeline settings.")
                image = pipe(prompt=prompt, height=h, width=w, num_inference_steps=steps, guidance_scale=cfg, generator=gen).images[0]

            fname = f"{task_id}.png"
            image.save(Path(OUTPUTS_DIR) / fname)

            status_file.write_text(json.dumps({"status": "completed", "files": [fname]}))
            print(f"[API] ✅ Task {task_id} completed.")
            
        except Exception as e:
            print(f"[API] ❌ ERROR generating {task_id}: {e}")
            status_file.write_text(json.dumps({"status": "error", "detail": str(e)}))
        
        finally:
            # THE HOLY GRAIL OF VRAM MANAGEMENT: Nuke the fragmented memory instantly.
            gc.collect()
            torch.cuda.empty_cache()
            print("[API] 🧹 GPU Memory Purged. Ready for next request.")

@app.get("/")
def health():
    return {"status": "Diffusers API is Alive", "model": "Z-Image-Turbo-SDNQ-int8"}

@app.post("/generate-async")
async def generate_async(
    background_tasks: BackgroundTasks,
    prompt: str = Form(...),
    negative_prompt: str = Form(""),
    num_inference_steps: int = Form(8),
    height: int = Form(1024),
    width: int = Form(1024),
    guidance_scale: float = Form(0.0),
    seed: int = Form(None),
    max_sequence_length: int = Form(512)
):
    task_id = uuid.uuid4().hex
    background_tasks.add_task(
        process_workflow_bg, 
        prompt, negative_prompt, num_inference_steps, cfg=guidance_scale, 
        w=width, h=height, seed=seed, max_seq_len=max_sequence_length, task_id=task_id
    )
    return {"task_id": task_id, "status": "queued"}

@app.get("/status/{task_id}")
def get_status(task_id: str):
    p = Path(OUTPUTS_DIR) / f"task_{task_id}.json"
    if not p.exists(): return {"status": "pending"}
    data = json.loads(p.read_text())
    if data.get("status") == "completed" and data.get("files"):
        img_path = Path(OUTPUTS_DIR) / data["files"][0]
        if img_path.exists():
            return StreamingResponse(open(img_path, "rb"), media_type="image/png")
    return data

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=8001)
'''

with open("diffusers_api.py", "w") as f:
    f.write(code_content)
print("✅ 'diffusers_api.py' created (INT8 + Manual Embedding Bypass).")

In [ ]:
# ------------------------------------------------------------------------------
# Cell 3: Start System (Diffusers API + Zrok Tunnel)
# ------------------------------------------------------------------------------
import subprocess
import sys
import time
import os
import re
import glob
import requests

FACADE_PORT = 8001
ZROK_BINARY = "./zrok"

_token_matches = glob.glob("/kaggle/input/**/.zrok_api_key", recursive=True)
ZROK_TOKEN_PATH = _token_matches[0] if _token_matches else None

def get_zrok_token():
    if ZROK_TOKEN_PATH and os.path.isfile(ZROK_TOKEN_PATH):
        with open(ZROK_TOKEN_PATH, "r") as f: return f.read().strip()
    return None

def force_zrok_auth(token):
    if not os.path.exists(ZROK_BINARY):
        print("⬇️ Downloading Zrok...")
        subprocess.run("wget -q https://github.com/openziti/zrok/releases/download/v1.1.3/zrok_1.1.3_linux_amd64.tar.gz -O zrok.tar.gz", shell=True)
        subprocess.run("tar -xzf zrok.tar.gz && chmod +x zrok", shell=True)
    subprocess.run(f"{ZROK_BINARY} disable", shell=True, stderr=subprocess.DEVNULL, stdout=subprocess.DEVNULL)
    res = subprocess.run(f'{ZROK_BINARY} enable --headless "{token}"', shell=True, capture_output=True, text=True)
    return res.returncode == 0

def start_zrok_tunnel_loop(port):
    share_proc = subprocess.Popen([ZROK_BINARY, "share", "public", f"localhost:{port}", "--headless"], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)
    for _ in range(10):
        time.sleep(3)
        res = subprocess.run([ZROK_BINARY, "overview"], capture_output=True, text=True)
        m = re.search(r'(https?://[a-z0-9\-\.]+\.zrok\.io)', res.stdout)
        if m: return m.group(1)
    return "Timeout"

# 1. CLEANUP
os.system("pkill -9 -f zrok")
os.system("pkill -9 -f diffusers_api.py")
time.sleep(2)

# 2. START API
print(f"🚀 Starting Diffusers API on port {FACADE_PORT}...")
print("   (Downloading INT8 model weights... please wait up to 3 mins on first run)")
api_log = open("/kaggle/working/api.log", "w")
api_proc = subprocess.Popen([sys.executable, "diffusers_api.py"], stdout=api_log, stderr=subprocess.STDOUT)

api_ready = False
for i in range(300): # Increased wait time slightly for the larger INT8 download
    try:
        if requests.get(f"http://127.0.0.1:{FACADE_PORT}/", timeout=2).status_code == 200:
            print(f"✅ API ready (after {i+1}s).")
            api_ready = True
            break
    except: pass
    time.sleep(1)

if not api_ready:
    print("\n❌ API failed to start. Logs:")
    os.system("tail -n 30 /kaggle/working/api.log")
    api_proc.terminate()
else:
    # 3. START ZROK
    token = get_zrok_token()
    if token and force_zrok_auth(token):
        url = start_zrok_tunnel_loop(FACADE_PORT)
        print(f"""
========================================================
⚡ Z-IMAGE-TURBO DIFFUSERS API LIVE
   Model : Disty0/Z-Image-Turbo-SDNQ-int8
   Memory: Auto CPU-Offload + GC Purge
   Prompts: Extended to 512 tokens
🌍 Public URL : {url}
========================================================
""")
        try:
            while True:
                if api_proc.poll() is not None:
                    print("❌ API crashed! Logs:")
                    os.system("tail -n 20 /kaggle/working/api.log")
                    break
                time.sleep(30)
        except KeyboardInterrupt:
            print("\n🛑 Shutting down...")
            api_proc.terminate()
            os.system("pkill -9 -f zrok")
    else:
        print("⚠️ Zrok auth failed.")